# MobiHet S-a: new FL screening
Private DAGSNet, no raw client data exchanged. Internal holdouts only.
Locked protocol: docs/research/pilot-new-fl.md. Unlimited-channel feasibility; does not certify finite contact.
10 metrics from per-client confusion: accuracy, precision/recall/F1 macro/micro/weighted. Single-label micro precision/recall/F1 and weighted recall equal accuracy. Final r20; no best checkpoint.

In [1]:
CODE = ['__init__.py', 'archs.py', 'channel.py', 'dagsnet.py', 'metrics.py', 'mobility.py', 'new_fl.py', 'new_methods.py', 'p2.py', 'pilot.py', 'prepare.py']

import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
n = torch.cuda.device_count()
print('torch', torch.__version__, 'gpus', [torch.cuda.get_device_name(i) for i in range(n)], flush=True)
assert n == 2 and all(torch.cuda.get_device_capability(i) == (7, 5) for i in range(n))
hits = [p.parent for p in Path('/kaggle/input').rglob('new_fl.py') if (p.parent / 'train_0_X.npy').exists()]
assert len(hits) == 1, hits
SRC, WORK, OUT = hits[0], Path('/kaggle/working/work'), Path('/kaggle/working/runs')
SOURCE_OUT = Path('/kaggle/working/source'); SOURCE_OUT.mkdir(exist_ok=True)
for sub in ('mh', 'knowledge', 'papers/pilot/fixture'):
    (WORK / sub).mkdir(parents=True, exist_ok=True)
code_manifest = json.loads((SRC / 'source_manifest.json').read_text())
for name, sha in code_manifest.items():
    assert hashlib.sha256((SRC / name).read_bytes()).hexdigest() == sha, ('source mismatch', name)
    shutil.copyfile(SRC / name, SOURCE_OUT / name)
for f in SRC.iterdir():
    if f.name in CODE:
        dst = WORK / 'mh' / f.name
    elif f.name == 'meta.json':
        dst = WORK / 'knowledge' / f.name
    else:
        dst = WORK / 'papers/pilot/fixture' / f.name
    shutil.copyfile(f, dst)
manifest = json.loads((SRC / 'manifest.json').read_text())
for name, sha in manifest['files'].items():
    assert hashlib.sha256((SRC / name).read_bytes()).hexdigest() == sha, ('fixture mismatch', name)
shutil.copyfile(SRC / 'source_manifest.json', SOURCE_OUT / 'source_manifest.json')
shutil.copyfile(SRC / 'manifest.json', SOURCE_OUT / 'fixture_manifest.json')
print('verified source and fixture', len(code_manifest), len(manifest['files']), flush=True)


torch 2.10.0+cu128 gpus ['Tesla T4', 'Tesla T4']
verified source and fixture 12 66


In [2]:
SEED = 43
METHODS = ['local', 'diag', 'kme_gauss', 'kme', 'dm_gauss', 'dm']

T0 = time.time(); OUT.mkdir(exist_ok=True)
queue = list(METHODS); slots = [0, 0, 1, 1]; running = {}; done = {}; seen = {}
while queue or running:
    for slot, gpu in enumerate(slots):
        if slot not in running and queue and time.time() - T0 < 2 * 3600:
            method = queue.pop(0); d = OUT / f'{method}_s{SEED}'; d.mkdir(exist_ok=True)
            env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), OMP_NUM_THREADS='1', MKL_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
            cmd = [sys.executable, '-m', 'mh.new_fl', '--arm', method, '--method', method, '--seed', str(SEED), '--out', str(OUT)]
            log = open(d / 'log.txt', 'w')
            running[slot] = (method, subprocess.Popen(cmd, cwd=WORK, env=env, stdout=log, stderr=subprocess.STDOUT), log)
            print('start', method, 'gpu', gpu, flush=True)
    for slot, (method, proc, log) in list(running.items()):
        f = OUT / f'{method}_s{SEED}' / 'log.txt'; lines = f.read_text().splitlines()
        for line in lines[seen.get(method, 0):]:
            if line.startswith(('setup', 'r', 'done', 'Traceback')) or 'Error' in line:
                print(method, '|', line, flush=True)
        seen[method] = len(lines)
        if proc.poll() is not None:
            done[method] = proc.returncode; log.close(); del running[slot]
    if queue and not running and time.time() - T0 >= 2 * 3600:
        break
    if queue or running:
        time.sleep(20)
summary = {m: {'rc': rc, 'complete': json.loads((OUT / f'{m}_s{SEED}' / 'result.json').read_text())['complete']
              if (OUT / f'{m}_s{SEED}' / 'result.json').exists() else False} for m, rc in done.items()}
(OUT / f'summary_s{SEED}.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary), flush=True)
shutil.rmtree(WORK)  # source retained separately; raw fixture is an input, never a run output
assert set(summary) == set(METHODS) and all(v['rc'] == 0 and v['complete'] for v in summary.values()), summary


start local gpu 0
start diag gpu 0
start kme_gauss gpu 1
start kme gpu 1
local | setup local 2.5s payload up/down [1280, 1280]
diag | setup diag 2.9s payload up/down [172160, 87680]
kme_gauss | setup kme_gauss 11.4s payload up/down [336000, 169600]
kme | setup kme 11.3s payload up/down [336000, 169600]
local | r1 local s43 group gp None loss 0.8013 83s
diag | r1 diag s43 group gp None loss 2.3421 83s
kme_gauss | r1 kme_gauss s43 group gp None loss 1.8165 80s
kme | r1 kme s43 group gp None loss 1.8318 80s
local | r2 local s43 group gp None loss 0.4341 70s
diag | r2 diag s43 group gp None loss 1.4700 70s
kme_gauss | r2 kme_gauss s43 group gp None loss 0.6234 80s
kme | r2 kme s43 group gp None loss 0.6276 78s
local | r3 local s43 group gp None loss 0.3311 68s
diag | r3 diag s43 group gp None loss 1.2515 69s
kme_gauss | r3 kme_gauss s43 group gp None loss 0.4292 80s
kme | r3 kme s43 group gp None loss 0.4301 79s
local | r4 local s43 group gp None loss 0.2687 68s
diag | r4 diag s43 group gp